<a href="https://colab.research.google.com/github/nainy-sara/convolutional-neural-networks/blob/main/cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import torch #pytorch
import torch.nn as nn
from skimage.data import shepp_logan_phantom
from google.colab.patches import cv2_imshow
import cv2

In [ ]:
true_np = shepp_logan_phantom()
true_np_scaled = true_np
noisy_np = np.random.poisson(true_np_scaled)

def cv2disp(name, image):
    """
    Displays an image using cv2_imshow, ensuring it's in a displayable format.
    Handles scaling to 0-255 and converting to uint8 for cv2_imshow.
    """
    # Normalize image values to 0-1 range based on its min/max, then scale to 0-255.
    # This ensures displayable contrast for various types of image data (e.g., kernels, intermediate activations).
    min_val = np.min(image)
    max_val = np.max(image)
    if (max_val - min_val) < 1e-6: # Avoid division by zero if image is constant
        image_to_display = np.zeros_like(image, dtype=np.uint8) # Display black if constant
    else:
        normalized_image = (image - min_val) / (max_val - min_val)
        image_to_display = (normalized_image * 255).astype(np.uint8)

    cv2_imshow(image_to_display)
    print(f"Displayed: {name}") # A small print to indicate what was displayed


nxd = true_np.shape[0]

# Initial display
print("--- Original Images ---")
print("True Image (scaled to 0-255):")
cv2disp('true', true_np_scaled)
print("Noisy Image (Poisson noise applied to scaled true image):")
cv2disp('noisy', noisy_np)

class Convolution_NxN(nn.Module):
    def __init__(self, kernal_size):
        super(Convolution_NxN, self).__init__()
        self.conv1kernal = nn.Conv2d(1, 1, kernal_size, padding=(int(kernal_size/2), int(kernal_size/2)), bias=False)
        # Original code had this commented out, keeping it so.
        # self.conv1kernal.weight.data.fill_(1.0/(kernal_size*kernal_size))

    def forward(self, x):
            x = self.conv1kernal(x)
            return x

noisy_torch    = torch.from_numpy(noisy_np).float().unsqueeze(0).unsqueeze(0)
true_torch  = torch.from_numpy(true_np_scaled).float().unsqueeze(0).unsqueeze(0)


# --- Fixed Kernel Convolution ---
print("\n--- Fixed Kernel Convolution ---")
conv_fix_kernal = Convolution_NxN(31)

kernal_torch_fixed = list(conv_fix_kernal.parameters())[0]
print("Fixed Kernel (initial random weights):")
# Use INTER_NEAREST for resizing kernels to avoid interpolation artifacts if visualising small kernel
cv2disp('kernel', cv2.resize(np.squeeze(kernal_torch_fixed.detach().numpy()), (nxd,nxd), interpolation = cv2.INTER_NEAREST))

conv_out_torch_fixed = conv_fix_kernal(noisy_torch)
conv_out_np_fixed = np.squeeze(conv_out_torch_fixed.detach().numpy())
print("Output of Fixed Kernel Convolution on Noisy Image:")
cv2disp('output of conv (fixed kernel)', conv_out_np_fixed)


# --- Trainable Kernel Convolution ---
print("\n--- Trainable Kernel Convolution (Single Layer) ---")
con_trained_kernal = Convolution_NxN(31)
loss_function   = nn.MSELoss()
optimiser   = torch.optim.Adam(con_trained_kernal.parameters(), lr=0.003)

display_interval = 100 # Display results every N epochs

print("Training single-layer convolutional kernel...")
for epoch in range(500):
    con_trained_kernal.train() # Set model to training mode
    output = con_trained_kernal(noisy_torch)
    loss = loss_function(output, true_torch)

    optimiser.zero_grad()
    loss.backward()
    optimiser.step()

    if (epoch + 1) % display_interval == 0 or epoch == 0 or epoch == 499:
        print(f"Epoch {epoch+1}/500 - Loss: {loss.item():.6f}")
        con_trained_kernal.eval() # Set model to evaluation mode for display
        with torch.no_grad(): # Disable gradient calculations for inference
            output_eval = con_trained_kernal(noisy_torch)
            kernal_torch_trained = list(con_trained_kernal.parameters())[0]
            print(f"  Trained Kernel (Epoch {epoch+1}):")
            cv2disp('trained kernel', cv2.resize(np.squeeze(kernal_torch_trained.detach().numpy()), (nxd,nxd), interpolation = cv2.INTER_NEAREST))

            cnn_output_np = np.squeeze(output_eval.detach().numpy())
            print(f"  Output after Trained Single Conv (Epoch {epoch+1}):")
            cv2disp('output after trained conv', cnn_output_np)
        con_trained_kernal.train() # Resume training mode


# --- Multi-layer CNN Model ---
print("\n--- Multi-layer CNN Model (Deep Denoising Network) ---")
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        self.CNN = nn.Sequential(
            nn.Conv2d(1, 8, 7, padding=(3,3)), nn.PReLU(),
            nn.Conv2d(8, 8, 7, padding=(3,3)), nn.PReLU(),
            nn.Conv2d(8, 8, 7, padding=(3,3)), nn.PReLU(),
            nn.Conv2d(8, 8, 7, padding=(3,3)), nn.PReLU(),
            nn.Conv2d(8, 1, 7, padding=(3,3)), nn.PReLU(), # Final layer outputs 1 channel
        )

    def forward(self,x):
            x = self.CNN(x)
            return x

cnn_to_train = CNN()
loss_function = nn.MSELoss()
optimiser = torch.optim.Adam(cnn_to_train.parameters(), lr= 0.003)

print("Training multi-layer CNN...")
for epoch in range(500):
    cnn_to_train.train() # Set model to training mode
    output = cnn_to_train(noisy_torch)
    loss = loss_function(output, true_torch)

    optimiser.zero_grad()
    loss.backward()
    optimiser.step()

    if (epoch + 1) % display_interval == 0 or epoch == 0 or epoch == 499:
        print(f"Epoch {epoch+1}/500 - Loss: {loss.item():.6f}")
        cnn_to_train.eval() # Set model to evaluation mode for display
        with torch.no_grad(): # Disable gradient calculations for inference
            output_eval = cnn_to_train(noisy_torch)
            cnn_output_np = np.squeeze(output_eval.detach().numpy())
            print(f"  Output of CNN (Epoch {epoch+1}):")
            cv2disp('output after trained CNN', cnn_output_np)
        cnn_to_train.train() # Resume training mode
